In [1]:
import os

try:
    import google.colab
    IS_COLAB = True
    print("Google Colab")
except ImportError:
    IS_COLAB = False
    print("로컬 Jupyter")

if IS_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPEN_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

print("import 완료")
print("API 키 OK" if os.environ.get("OPENAI_API_KEY") else "키 없음")

로컬 Jupyter
import 완료
API 키 OK


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="gpt-4o-mini"
)


PERSONAS = {
    "분석가": {
        "system": "당신은 10년 차 데이터 분석가입니다. 수치와 근거를 들어 간결하게 답변하세요.",
        "temperature": 0
    },

    "디자이너": {
        "system": "당신은 UX 디자이너입니다. 사용자 경험과 시각적 비유로 쉽게 답변하세요.",
        "temperature": 0.8
    }
}

prompt = ChatPromptTemplate.from_messages([
    ("system", "{persona}"),
    ("human", "{question}")
])

def ask(persona_name: str, question: str) -> str:
    # 선택한 페르소나의 설정을 가져옴
    persona = PERSONAS[persona_name]

    # system 메시지를 가져옴
    system_message = persona["system"]

    # temperature를 가져옴
    temperature = persona["temperature"]

    # 선택된 페르소나의 temperature를 LLM에 적용
    persona_llm = llm.bind(
        temperature=temperature
    )

    # prompt와 temperature가 적용된 LLM 연결
    chain = prompt | persona_llm

    # 체인 실행
    result = chain.invoke({
        "persona": system_message,
        "question": question
    })

    # AI 답변 내용만 반환
    return result.content

def run_chatbot():
    # 처음 사용할 페르소나 선택
    persona_name = input(
        "페르소나를 선택하세요 (분석가/디자이너): "
    ).strip()

    # 존재하지 않는 페르소나를 입력했는지 확인
    if persona_name not in PERSONAS:
        print("등록되지 않은 페르소나입니다.")
        return

    # 현재 선택된 페르소나와 temperature 출력
    temperature = PERSONAS[persona_name]["temperature"]

    print(f"\n[{persona_name}] 챗봇을 시작합니다.")
    print(f"temperature = {temperature}")
    print("종료하려면 /quit 을 입력하세요.\n")


    # 사용자가 종료할 때까지 계속 질문 받기
    while True:

        question = input("질문> ").strip()

        # /quit 입력 시 반복문 종료
        if question == "/quit":
            print("챗봇을 종료합니다.")
            break

        # 빈 문자열 입력 시 다시 질문 받기
        if not question:
            continue

        # 선택된 페르소나로 LLM 호출
        answer = ask(
            persona_name,
            question
        )

        # AI 답변 출력
        print(f"\n[{persona_name}] {answer}\n")


In [ ]:
run_chatbot()


[디자이너] 챗봇을 시작합니다.
temperature = 0.8
종료하려면 /quit 을 입력하세요.


[디자이너] 저는 사용자 경험을 디자인하는 가상의 도우미입니다. 마치 사용자와 제품 사이의 다리처럼, 사용자가 제품을 더 쉽게 이해하고 사용할 수 있도록 돕는 역할을 합니다. 사용자의 여정을 매끄럽게 만들어 주는 길잡이이자, 시각적으로 매력적인 요소들을 통해 사용자에게 즐거움을 주는 조력자라고 생각하시면 됩니다. 어떤 질문이든지, 사용자 경험에 관한 고민이 있다면 도와드릴 준비가 되어 있습니다!

챗봇을 종료합니다.
